### Type 1

In [ ]:
import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
df = pd.read_csv(r'Data\DataCleaned\master_cleaned.csv')

In [ ]:
df.head()

,itemdesc,company,brand,packaging,qty,uomdesc,pack_size
0,ELEPHANT APPLE SODA-PET 500 ML,CEYLON COLD STORES,ELEPHANT APPLE SODA,PET,500,ML,500.00 ML
1,CAMEL ASTO-PET 1500 ML,ARTINAT CORDIALS INDUSTRIES,CAMEL ASTO,PET,1500,ML,1500.00 ML
2,CAMEL ASTO-PET 400 ML,ARTINAT CORDIALS INDUSTRIES,CAMEL ASTO,PET,400,ML,400.00 ML
3,CAMEL ASTO-PET 750 ML,ARTINAT CORDIALS INDUSTRIES,CAMEL ASTO,PET,750,ML,750.00 ML
4,ELEPHANT NECTO-CAN 330 ML,CEYLON COLD STORES,ELEPHANT NECTO,CAN,330,ML,330.00 ML


In [ ]:
# List unique items in 'packaging' and 'uomdesc' columns
unique_packaging = df['packaging'].unique()
unique_uomdesc = df['qty'].unique()

print("Unique items in 'packaging':", unique_packaging)
print("Unique items in 'uomdesc':", unique_uomdesc)

In [ ]:
# Get the datatype of each column
column_dtypes = df.dtypes
print(column_dtypes)

itemdesc     object
company      object
brand        object
packaging    object
qty           int64
uomdesc      object
pack_size    object
dtype: object


### **Data Preprocessing for the Labelled Data**

#### **Before Master Cleaning** i.e during June

In [ ]:
# pip install openpyxl

In [ ]:
import pandas as pd

df = pd.read_excel(r'Labelled_Data\Data.xlsx')

In [ ]:
df.head()

,PERIOD,AUDITTYPE,STORECODE,DLRCODE,ITEMCODE,CATEGORY,MANUFACTURE,BRAND,ITEMDESC,MRP,...,Matched: ITEMDESC,Matched: BRAND,Matched: MANUFACTURE,Matched: PACKTYPE,Matched: PACKSIZE,Reason,Suggestion,Score,Datacore Matching,Datacore Matching Details
0,202410,1,120547179,10946010003,1730008019565,14,SMITHKLINE BEECHAM (PVT) LTD,SENSODYNE,SENSODYNE SOFT 1 NO SAVE 95/=,195,...,NaN,NaN,NaN,NaN,NaN,Target Company not found| Target Brand not fou...,NaN,NaN,Wrong,Wrong No Match
1,202410,1,133670987,19354010001,1729584139818,12,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA/ PLPCH/ 100GM,290,...,NaN,NaN,DAINTEE MARKETING,NaN,NaN,| Target Brand not found| Target Packtype not ...,NaN,NaN,Correct,Correct No Match
2,202410,1,133670987,19354010001,1729584017998,12,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA /PLPCH /50GM,155,...,NaN,NaN,DAINTEE MARKETING,NaN,NaN,| Target Brand not found| Target Packtype not ...,NaN,NaN,Correct,Correct No Match
3,202410,1,53951366,10006010001,1729575545353,26,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO WITH ...,340,...,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO WITH ...,SUNSILK,UNILEVER SRI LANKA LTD,PLBOT,80.00 ML,NaN,EMF,0.964578,Wrong,Wrong Match
4,202410,1,53951366,10006010001,1729572854804,10,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM SAVE 10/=,150,...,FREELAN MALDIVE FISH FLAVOUR 70 GM,FREELAN,FREELAN ENTERPRISES,PLPCH,70.00 GM,NaN,EMF,0.963639,Correct,Correct Match


In [ ]:
transaction = df.iloc[:, [6, 7, 8, 10, 11]]
transaction.to_csv(r'Labelled_Data\transaction.csv', index=False)

master = df.iloc[:, [15, 22, 25, 26, 27, 30, 31, 32]]
master.to_csv(r'Labelled_Data\master.csv', index=False)

##### **Making the Transaction File to the Desired Format**

In [ ]:
transaction.head()

,MANUFACTURE,BRAND,ITEMDESC,PACKSIZE,PACKTYPE
0,SMITHKLINE BEECHAM (PVT) LTD,SENSODYNE,SENSODYNE SOFT 1 NO SAVE 95/=,1 NO,BRUSH
1,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA/ PLPCH/ 100GM,100GM,PLPCH
2,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA /PLPCH /50GM,50GM,PLPCH
3,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO WITH ...,80 ML,PLBOT
4,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM SAVE 10/=,70 GM,PLPCH


In [ ]:
import pandas as pd
import re

# Load the CSV
df = pd.read_csv(r"Labelled_Data\transaction.csv")

# Step 1: Drop specific columns (ensure exact match, ignore if not found)
# columns_to_drop = ['PERIOD','AUDITTYPE','STORECODE','DLRCODE','ITEMCODE','CATEGORY','MRP','COMMENTS','IMAGE']
# df = df.drop(columns=columns_to_drop, errors='ignore')

# Step 2: Drop existing QTY and UNIT if they already exist to avoid duplicates
# for col in ['QTY', 'UNIT']:
#     if col in df.columns:
#         df = df.drop(columns=col)

# Step 3: Extract QTY and UNIT from PACKSIZE
def extract_qty_unit(packsize):
    if pd.isna(packsize) or not isinstance(packsize, str) or packsize.strip() == "":
        return pd.Series([None, None])

    match = re.match(r'(\d+(?:\.\d+)?)\s*([A-Za-z]*)', packsize.strip())
    if match:
        qty = match.group(1)
        unit = match.group(2).upper() if match.group(2) else None
        return pd.Series([qty, unit])
    else:
        return pd.Series([None, None])

# Apply the extraction function to PACKSIZE
df[['QTY', 'UNIT']] = df['PACKSIZE'].apply(extract_qty_unit)

# Step 4: Normalize UNIT values
df['UNIT'] = df['UNIT'].replace({'G': 'GM', 'L': 'LTR'})

# Step 5: Replace null/empty values in QTY, UNIT, PACKSIZE, PACKTYPE with '10000'
# for col in ['QTY', 'UNIT', 'PACKSIZE', 'PACKTYPE']:
#     df[col] = df[col].fillna('10000')
#     df[col] = df[col].replace('', '10000')

# Step 6: Reorder to place QTY and UNIT just before PACKSIZE
cols = df.columns.tolist()
if 'PACKSIZE' in cols:
    # Remove QTY and UNIT if already present elsewhere
    cols = [c for c in cols if c not in ['QTY', 'UNIT']]
    packsize_index = cols.index('PACKSIZE')
    new_order = cols[:packsize_index] + ['QTY', 'UNIT'] + cols[packsize_index:]
    df = df[new_order]

# Save cleaned file
df.to_csv(r"Labelled_Data\transaction.csv", index=False)

In [ ]:
# Transaction
df.head()

,MANUFACTURE,BRAND,ITEMDESC,QTY,UNIT,PACKSIZE,PACKTYPE
0,SMITHKLINE BEECHAM (PVT) LTD,SENSODYNE,SENSODYNE SOFT 1 NO SAVE 95/=,1,NO,1 NO,BRUSH
1,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA/ PLPCH/ 100GM,100,GM,100GM,PLPCH
2,IDEA AFFIX MARKETING,IDEA TEA,IDEA TEA /PLPCH /50GM,50,GM,50GM,PLPCH
3,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO WITH ...,80,ML,80 ML,PLBOT
4,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM SAVE 10/=,70,GM,70 GM,PLPCH


In [ ]:
# MASTER

df.head()

,nitemcode,company,brand,itemdesc,qty,uomdesc,pack_size,packaging
0,63964,GLAXO SMITHKLINE BEECHAM,SENSODYNE,SENSODYNE - SOFT TOOTHBRUSH 1 NO,1,NO,1.00 NO,NaN
3,63972,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO 5 NAT...,80,ML,80.00 ML,PLBOT
4,34284,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM,70,GM,70.00 GM,PLPCH
5,63973,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK STUNNING BLACK SHINE-GO SHINE-WITH AML...,80,ML,80.00 ML,PLBOT
6,63955,PELWATTE DAIRY INDUSTRIES (PVT) LTD,PELWATTE,PELWATTE FULL CREAM MILK POWDER 400 GM PLPCH +...,400,GM,400.00 GM,PLPCH


##### **Making the Master file to the Desired Foramt**

In [ ]:
master.head()

,NITEMCODE,Master: itemdesc,Master: company,Master: brand,Master: packaging,Master: qty,Master: uomdesc,Master: pack_size
0,63964,SENSODYNE - SOFT TOOTHBRUSH 1 NO,GLAXO SMITHKLINE BEECHAM,SENSODYNE,NaN,1.0,NO,1.00 NO
1,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,63972,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO 5 NAT...,UNILEVER SRI LANKA LTD,SUNSILK,PLBOT,80.0,ML,80.00 ML
4,34284,FREELAN MALDIVE FISH FLAVOUR 70 GM,FREELAN ENTERPRISES,FREELAN,PLPCH,70.0,GM,70.00 GM


In [ ]:
import pandas as pd

# Load the CSV
df = pd.read_csv(r"Labelled_Data\master.csv")

# Step 1: Rename columns by removing 'Master: ' prefix and convert to lowercase
df.columns = [col.replace('Master: ', '').strip().lower() for col in df.columns]

# Step 2: Drop rows where NITEMCODE is 9
if 'nitemcode' in df.columns:
    df = df[df['nitemcode'] != 9]

# Step 3: Replace G -> GM and L -> LTR in uomdesc (case-sensitive)
if 'uomdesc' in df.columns:
    df['uomdesc'] = df['uomdesc'].replace({'G': 'GM', 'L': 'LTR'})

# Step 4: Replace nulls or empty strings with 'NOT' in specified columns
for col in ['qty', 'uomdesc', 'pack_size']:
    if col in df.columns:
        df[col] = df[col].fillna('NOT')
        df[col] = df[col].replace('', 'NOT')

# Step 5: Remove .0 from qty column
if 'qty' in df.columns:
    def clean_qty(val):
        try:
            return int(float(val))
        except:
            return val  # Leave as-is if it can't be converted
    df['qty'] = df['qty'].apply(clean_qty)

# Step 6: Reorder columns to match desired sequence
desired_order = ['nitemcode', 'company', 'brand', 'itemdesc', 'qty', 'uomdesc', 'pack_size', 'packaging']
df = df[[col for col in desired_order if col in df.columns]]

# # Step 7: Optional: Save backup copy before final output
# df.to_csv(r"Labelled_Data\master_backup.csv", index=False)

# Save cleaned file
df.to_csv(r"Labelled_Data\master.csv", index=False)

In [ ]:
df.head()

,nitemcode,company,brand,itemdesc,qty,uomdesc,pack_size,packaging
0,63964,GLAXO SMITHKLINE BEECHAM,SENSODYNE,SENSODYNE - SOFT TOOTHBRUSH 1 NO,1,NO,1.00 NO,NaN
3,63972,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK NOURISHING SOFT & SMOOTH SHAMPOO 5 NAT...,80,ML,80.00 ML,PLBOT
4,34284,FREELAN ENTERPRISES,FREELAN,FREELAN MALDIVE FISH FLAVOUR 70 GM,70,GM,70.00 GM,PLPCH
5,63973,UNILEVER SRI LANKA LTD,SUNSILK,SUNSILK STUNNING BLACK SHINE-GO SHINE-WITH AML...,80,ML,80.00 ML,PLBOT
6,63955,PELWATTE DAIRY INDUSTRIES (PVT) LTD,PELWATTE,PELWATTE FULL CREAM MILK POWDER 400 GM PLPCH +...,400,GM,400.00 GM,PLPCH


#### **After Structuring** i.e during August

In [1]:
import pandas as pd

master_df = pd.read_csv('master.csv')
transaction_df = pd.read_csv('transaction.csv')

In [2]:
master_df.head()

,itemcode,itemdesc,catcode,category,company,brand,packaging,flavor,color,qty,uomdesc,pack_size,launchdate
0,46118,WINSTON PURPLE MINT-WINSTON-FRESH-HL 20'S-KS R...,105,CIGARETTE,JT INTERNATIONAL - SA,WINSTON PURPLE MINT,HL,FRESH,NaN,20.0,NO,20,201707.0
1,90110250,MARLBORO GOLD-MARLBORO-LIGHT-HL 20'S-KS REGULA...,105,CIGARETTE,PT PHILIP MORRIS-INDONESIA,MARLBORO GOLD,HL,LIGHT,NaN,20.0,NO,20,201505.0
2,90110225,SAHASI PREMIUM FILTER KINGS-SAHASHI-FULL FLAVO...,105,CIGARETTE,CHAUDHARY GROUP - NEPAL,SAHASI PREMIUM FILTER KINGS,HL,FULL FLAVOUR,NaN,20.0,NO,20,201102.0
3,90110167,GAURAV-GAURAV-FULL FLAVOUR-SC 20'S-KS REGULAR-...,105,CIGARETTE,CHAUDHARY GROUP - NEPAL,GAURAV,SC,FULL FLAVOUR,NaN,20.0,NO,20,0.0
4,90110215,BROWN CLOVE-BROWN-KRETEK-HL 20'S-LONG REGULAR-...,105,CIGARETTE,GORKHA LAHARI PVT.LTD.- NEPAL,BROWN CLOVE,HL,KRETEK,NaN,20.0,NO,20,200801.0


In [3]:
transaction_df.head()

,PERIOD,AUDITTYPE,STORECODE,DLRCODE,ITEMCODE,CATEGORY,MANUFACTURE,BRAND,ITEMDESC,MRP,PACKSIZE,PACKTYPE,COMMENTS,IMAGE
0,202405,2,129995584,21202020023,1716875204553,134,LIFE FOOD & BEVERAGE-NEPAL,FARM FRESH COCO,FARM FRESH COCO \180ML\CAN\COCONUT FLAVOUR,80.0,180ML,CAN,NaN,NaN
1,202405,2,111850613,31304020009,1717055408953,118,VARUN BEVERAGE -NEPAL,MIRINDA ORANGE,MIRINDA ORANGE PBT 2.25 L 250ML MOUNTAIN DEW FREE,270.0,2.25L,PBT,NaN,1717055408953-varun beverage -nepal-mirinda or...
2,202405,2,116587488,41301020009,1716958991319,134,R.R PROCESSED DRINKING WATER,R.R DRINKING WATER,R.R,30.0,1000ML,PET,NaN,NaN
3,202405,2,112206294,41301020022,1716952566648,134,BAUDDHA GRAMEEN PROCESSED DRINKING WATER,PROCESSED DRINKING WATER,BG AQUA FRESH,30.0,1000ML,PET,NaN,NaN
4,202405,2,112313594,41301020012,1716952566648,134,BAUDDHA GRAMEEN PROCESSED DRINKING WATER,PROCESSED DRINKING WATER,BG AQUA FRESH,30.0,1000ML,PET,NaN,NaN


In [4]:
print(master_df.columns.to_list())
print(transaction_df.columns.to_list())

['itemcode', 'itemdesc', 'catcode', 'category', 'company', 'brand', 'packaging', 'flavor', 'color', 'qty', 'uomdesc', 'pack_size', 'launchdate']
['PERIOD', 'AUDITTYPE', 'STORECODE', 'DLRCODE', 'ITEMCODE', 'CATEGORY', 'MANUFACTURE', 'BRAND', 'ITEMDESC', 'MRP', 'PACKSIZE', 'PACKTYPE', 'COMMENTS', 'IMAGE']


In [5]:
# PRINT ALL UNIQUE VALUES
# for col in master_df.columns:
#   print(f"Unique values in '{col}':")
#   display(master_df[col].unique())
#   print("-" * 30)

print(master_df['uomdesc'].unique())
print(transaction_df['PACKSIZE'].unique())

['NO' nan 'ML' 'GM']
['180ML' '2.25L' '1000ML' '1.5LTR' '2.25LTR' '125ML' '250ML' '110ML' '600'
 '180 ML' '250ML+15ML FREE' '330ML' '320ML' '240ML' '150ML' '500' '250'
 '200ML' '135ML' '1750ML' '160ML ' '65 ML.' '180 ML.' nan '500ML' '119 ML'
 '1500ML ']


In [6]:
import pandas as pd
import numpy as np
import re

# Example: transaction_df is already loaded

# Step 1: Identify rows with invalid characters (anything other than digits, letters, spaces, or '.')
pattern_valid = r'^[0-9. ]*[A-Za-z ]*[A-Za-z]*$'
mask_valid = transaction_df['PACKSIZE'].isna() | transaction_df['PACKSIZE'].astype(str).str.match(pattern_valid)

# dropped rows
dropped_trans = transaction_df[~mask_valid].copy()

# keep only valid rows
transaction = transaction_df[mask_valid].copy()

# Step 2: Extract qty and uomdesc
def split_packsize(val):
    if pd.isna(val):
        return np.nan, np.nan
    # remove extra spaces and dots at end
    val = str(val).strip()
    # extract numeric part
    qty_match = re.match(r'([0-9.]+)', val)
    qty = float(qty_match.group(1)) if qty_match else np.nan
    # extract unit part
    uom_match = re.search(r'([A-Za-z]+)', val)
    uom = uom_match.group(1).upper() if uom_match else np.nan
    return qty, uom

transaction[['qty', 'uomdesc']] = transaction['PACKSIZE'].apply(lambda x: pd.Series(split_packsize(x)))

# Reset index
transaction.reset_index(drop=True, inplace=True)
dropped_trans.reset_index(drop=True, inplace=True)

In [7]:
dropped_trans.shape

(11, 14)

In [8]:
transaction.head()

,PERIOD,AUDITTYPE,STORECODE,DLRCODE,ITEMCODE,CATEGORY,MANUFACTURE,BRAND,ITEMDESC,MRP,PACKSIZE,PACKTYPE,COMMENTS,IMAGE,qty,uomdesc
0,202405,2,129995584,21202020023,1716875204553,134,LIFE FOOD & BEVERAGE-NEPAL,FARM FRESH COCO,FARM FRESH COCO \180ML\CAN\COCONUT FLAVOUR,80.0,180ML,CAN,NaN,NaN,180.00,ML
1,202405,2,111850613,31304020009,1717055408953,118,VARUN BEVERAGE -NEPAL,MIRINDA ORANGE,MIRINDA ORANGE PBT 2.25 L 250ML MOUNTAIN DEW FREE,270.0,2.25L,PBT,NaN,1717055408953-varun beverage -nepal-mirinda or...,2.25,L
2,202405,2,116587488,41301020009,1716958991319,134,R.R PROCESSED DRINKING WATER,R.R DRINKING WATER,R.R,30.0,1000ML,PET,NaN,NaN,1000.00,ML
3,202405,2,112206294,41301020022,1716952566648,134,BAUDDHA GRAMEEN PROCESSED DRINKING WATER,PROCESSED DRINKING WATER,BG AQUA FRESH,30.0,1000ML,PET,NaN,NaN,1000.00,ML
4,202405,2,112313594,41301020012,1716952566648,134,BAUDDHA GRAMEEN PROCESSED DRINKING WATER,PROCESSED DRINKING WATER,BG AQUA FRESH,30.0,1000ML,PET,NaN,NaN,1000.00,ML


In [9]:
print(master_df.columns.to_list())
print(transaction.columns.to_list())

['itemcode', 'itemdesc', 'catcode', 'category', 'company', 'brand', 'packaging', 'flavor', 'color', 'qty', 'uomdesc', 'pack_size', 'launchdate']
['PERIOD', 'AUDITTYPE', 'STORECODE', 'DLRCODE', 'ITEMCODE', 'CATEGORY', 'MANUFACTURE', 'BRAND', 'ITEMDESC', 'MRP', 'PACKSIZE', 'PACKTYPE', 'COMMENTS', 'IMAGE', 'qty', 'uomdesc']


In [10]:
master = master_df[['itemcode', 'itemdesc', 'catcode', 'company', 'brand', 'packaging', 'qty', 'uomdesc']].copy()
master = master[['catcode', 'company', 'brand', 'itemdesc', 'packaging', 'qty', 'uomdesc']]
transaction = transaction[['CATEGORY', 'MANUFACTURE', 'BRAND', 'ITEMDESC', 'PACKTYPE', 'qty', 'uomdesc']].copy()

# Optional: Make all column names lowercase for consistency (optional but often helpful)
# master.columns = master.columns.str.lower()
# transaction.columns = transaction.columns.str.lower()

In [11]:
master.head()

,catcode,company,brand,itemdesc,packaging,qty,uomdesc
0,105,JT INTERNATIONAL - SA,WINSTON PURPLE MINT,WINSTON PURPLE MINT-WINSTON-FRESH-HL 20'S-KS R...,HL,20.0,NO
1,105,PT PHILIP MORRIS-INDONESIA,MARLBORO GOLD,MARLBORO GOLD-MARLBORO-LIGHT-HL 20'S-KS REGULA...,HL,20.0,NO
2,105,CHAUDHARY GROUP - NEPAL,SAHASI PREMIUM FILTER KINGS,SAHASI PREMIUM FILTER KINGS-SAHASHI-FULL FLAVO...,HL,20.0,NO
3,105,CHAUDHARY GROUP - NEPAL,GAURAV,GAURAV-GAURAV-FULL FLAVOUR-SC 20'S-KS REGULAR-...,SC,20.0,NO
4,105,GORKHA LAHARI PVT.LTD.- NEPAL,BROWN CLOVE,BROWN CLOVE-BROWN-KRETEK-HL 20'S-LONG REGULAR-...,HL,20.0,NO


In [12]:
transaction.head()

,CATEGORY,MANUFACTURE,BRAND,ITEMDESC,PACKTYPE,qty,uomdesc
0,134,LIFE FOOD & BEVERAGE-NEPAL,FARM FRESH COCO,FARM FRESH COCO \180ML\CAN\COCONUT FLAVOUR,CAN,180.00,ML
1,118,VARUN BEVERAGE -NEPAL,MIRINDA ORANGE,MIRINDA ORANGE PBT 2.25 L 250ML MOUNTAIN DEW FREE,PBT,2.25,L
2,134,R.R PROCESSED DRINKING WATER,R.R DRINKING WATER,R.R,PET,1000.00,ML
3,134,BAUDDHA GRAMEEN PROCESSED DRINKING WATER,PROCESSED DRINKING WATER,BG AQUA FRESH,PET,1000.00,ML
4,134,BAUDDHA GRAMEEN PROCESSED DRINKING WATER,PROCESSED DRINKING WATER,BG AQUA FRESH,PET,1000.00,ML


In [21]:
# print("Master Packaging: ", master['packaging'].unique())
# print("Transa Packaging: ", transaction['PACKTYPE'].unique())
# print("Transa uomdesc  : ", transaction['uomdesc'].unique())
# print("Transa qty      : ", transaction['qty'].unique())

Master Packaging:  ['HL' 'SC' 'SB' nan 'RGB' 'PET' 'CAN' 'PLAST' 'TIN C' 'TPK' 'JAR' 'PPH']
Transa Packaging:  ['CAN' 'PBT' 'PET' 'PET ' 'TPK' nan]
Transa uomdesc  :  ['ML' 'L' 'LTR' nan]
Transa qty      :  [1.80e+02 2.25e+00 1.00e+03 1.50e+00 1.25e+02 2.50e+02 1.10e+02 6.00e+02
 3.30e+02 3.20e+02 2.40e+02 1.50e+02 5.00e+02 2.00e+02 1.35e+02 1.75e+03
 1.60e+02      nan 1.19e+02 1.50e+03]


In [13]:
import numpy as np

# Step 1: Remove trailing spaces from PACKTYPE
transaction['PACKTYPE'] = transaction['PACKTYPE'].astype(str).str.strip()

# Step 2: Standardize UOM and convert qty accordingly
def convert_uom(row):
    if pd.isna(row['uomdesc']):
        return row['qty'], row['uomdesc']  # leave NaNs as is

    uom = row['uomdesc'].strip().upper()
    qty = row['qty']

    if uom == 'L' or uom == 'LTR':
        return qty * 1000, 'ML'
    elif uom == 'KG':
        return qty * 1000, 'GM'
    else:
        return qty, uom  # no conversion

transaction[['qty', 'uomdesc']] = transaction.apply(lambda r: pd.Series(convert_uom(r)), axis=1)

# Save changes back to transaction
transaction.reset_index(drop=True, inplace=True)

# Check result
# print(transaction['PACKTYPE'].unique())
# print(transaction['uomdesc'].unique())
# print(transaction.head())

In [14]:
print(transaction['PACKTYPE'].unique())
print(transaction['uomdesc'].unique())

['CAN' 'PBT' 'PET' 'TPK' 'nan']
['ML' nan]


In [15]:
transaction.head()

,CATEGORY,MANUFACTURE,BRAND,ITEMDESC,PACKTYPE,qty,uomdesc
0,134,LIFE FOOD & BEVERAGE-NEPAL,FARM FRESH COCO,FARM FRESH COCO \180ML\CAN\COCONUT FLAVOUR,CAN,180.0,ML
1,118,VARUN BEVERAGE -NEPAL,MIRINDA ORANGE,MIRINDA ORANGE PBT 2.25 L 250ML MOUNTAIN DEW FREE,PBT,2250.0,ML
2,134,R.R PROCESSED DRINKING WATER,R.R DRINKING WATER,R.R,PET,1000.0,ML
3,134,BAUDDHA GRAMEEN PROCESSED DRINKING WATER,PROCESSED DRINKING WATER,BG AQUA FRESH,PET,1000.0,ML
4,134,BAUDDHA GRAMEEN PROCESSED DRINKING WATER,PROCESSED DRINKING WATER,BG AQUA FRESH,PET,1000.0,ML


In [16]:
master.head()

,catcode,company,brand,itemdesc,packaging,qty,uomdesc
0,105,JT INTERNATIONAL - SA,WINSTON PURPLE MINT,WINSTON PURPLE MINT-WINSTON-FRESH-HL 20'S-KS R...,HL,20.0,NO
1,105,PT PHILIP MORRIS-INDONESIA,MARLBORO GOLD,MARLBORO GOLD-MARLBORO-LIGHT-HL 20'S-KS REGULA...,HL,20.0,NO
2,105,CHAUDHARY GROUP - NEPAL,SAHASI PREMIUM FILTER KINGS,SAHASI PREMIUM FILTER KINGS-SAHASHI-FULL FLAVO...,HL,20.0,NO
3,105,CHAUDHARY GROUP - NEPAL,GAURAV,GAURAV-GAURAV-FULL FLAVOUR-SC 20'S-KS REGULAR-...,SC,20.0,NO
4,105,GORKHA LAHARI PVT.LTD.- NEPAL,BROWN CLOVE,BROWN CLOVE-BROWN-KRETEK-HL 20'S-LONG REGULAR-...,HL,20.0,NO


In [17]:
master.to_csv('master_cleaned.csv', index=False)
transaction.to_csv('transaction_cleaned.csv', index=False)